<a href="https://colab.research.google.com/github/rajashekar-0111/python-project-sem3/blob/main/Copy_of_cosine_0_4_lasso_regressior.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

# --- 1. Load data ---
df1 = pd.read_csv('Tuesday-WorkingHours.pcap_ISCX.csv', low_memory=True)
df2 = pd.read_csv('Wednesday-workingHours.pcap_ISCX.csv', low_memory=True)
df3 = pd.read_csv('Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', low_memory=True, on_bad_lines='skip')

dataset = pd.concat([df1, df2, df3], ignore_index=True)
dataset.columns = dataset.columns.str.strip()

#----------------------

X = dataset.iloc[:, :-1]
y = dataset.iloc[:, -1]

X = X.apply(pd.to_numeric, errors='coerce')
X.replace([np.inf, -np.inf], np.nan, inplace=True)

#------------------

from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
X = imputer.fit_transform(X)

from sklearn.preprocessing import LabelEncoder
labelencoder_y = LabelEncoder()
y = labelencoder_y.fit_transform(y)

#------------------------
# Filter out classes with only one member to prevent ValueError in initial stratified split
class_counts = pd.Series(y).value_counts()
single_member_classes = class_counts[class_counts == 1].index

if not single_member_classes.empty:
    mask = ~pd.Series(y).isin(single_member_classes)
    X = X[mask]
    y = y[mask]

# --- 2. SUBSAMPLE ---
# Kernel PCA needs an n x n matrix in memory, so the full ~1.3M rows
# is impossible. This keeps 15,000 rows with the same class balance.
from sklearn.model_selection import train_test_split
X, _, y, _ = train_test_split(X, y, train_size=15000, random_state=0, stratify=y)

# Filter out classes with only one member *again* because subsampling might have reduced some counts to 1
sub_class_counts = pd.Series(y).value_counts()
sub_single_member_classes = sub_class_counts[sub_class_counts < 2].index

if not sub_single_member_classes.empty:
    mask = ~pd.Series(y).isin(sub_single_member_classes)
    X = X[mask]
    y = y[mask]

# --- 3. Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.4,        # change to 0.2 or 0.6 for other runs
    random_state=0,
    stratify=y
)

#-----------------------------------
# --- 4. Kernel PCA ---
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(n_components=2, kernel='cosine', gamma=15)   # swap kernel/gamma per run
X_train = kpca.fit_transform(X_train)
X_test = kpca.transform(X_test)

#---------------------------------------------------------------
# --- 5. Classifier (swap this block for each algorithm) ---
from sklearn.linear_model import Lasso
regressor = Lasso(alpha=0.1)
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_test)

#___________________


# Convert continuous Linear Regression output into class labels
y_pred = np.rint(y_pred).astype(int)

# Keep predictions within the valid class range
y_pred = np.clip(y_pred, y.min(), y.max())
#---------------------------------------------------------------

# --- 6. Metrics ---
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n")
print(cm)

print("\nAccuracy : {:.4f}".format(accuracy_score(y_test, y_pred)))
print("\nPrecision : {:.4f}".format(precision_score(y_test, y_pred, average='weighted')))
print("\nRecall : {:.4f}".format(recall_score(y_test, y_pred, average='weighted')))
print("\nF1 Score : {:.4f}".format(f1_score(y_test, y_pred, average='weighted')))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Confusion Matrix:

[[4561    0    0    0    0    0    0    0]
 [1222    0    0    0    0    0    0    0]
 [  49    0    0    0    0    0    0    0]
 [  52    0    0    0    0    0    0    0]
 [  70    0    0    0    0    0    0    0]
 [  28    0    0    0    0    0    0    0]
 [  13    0    0    0    0    0    0    0]
 [   5    0    0    0    0    0    0    0]]

Accuracy : 0.7602

Precision : 0.5779

Recall : 0.7602

F1 Score : 0.6566

Classification Report:

              precision    recall  f1-score   support

           0       0.76      1.00      0.86      4561
           1       0.00      0.00      0.00      1222
           2       0.00      0.00      0.00        49
           3       0.00      0.00      0.00        52
           4       0.00      0.00      0.00        70
           5       0.00      0.00      0.00        28
           6       0.00      0.00      0.00        13
           8       0.00      0.00      0.00         5

    accuracy                           0.76     

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/m